# LiCSBAS Jakarta v13 — preregistered descending shorter-window inversion

V11 failed because the public descending catalogue ended on 11 June 2023 and could not span 95% of the originally requested 2020–2023 interval. Its retained network itself was strong, and the subsequent descriptive audit found that 68.39% of North Jakarta passed every non-temporal local requirement. V13 therefore asks a new, narrower question: can the exact available interval, **10 January 2020–11 June 2023**, produce a defensible North-Jakarta velocity candidate?

This is not a retroactive v11 pass. The selected interval and pre-inversion network behaviour are already known and are recorded as such. The prospective outcome is the post-inversion mask. A candidate pixel must have finite velocity and velocity uncertainty, satisfy the unchanged local network criteria, and have both `n_gap=0` and `n_loop_err=0`. Continuation still requires at least 50% qualifying North area, at least 25% in every northern district, and at least 75% of qualifying area in one eight-connected patch. Uncertainty survival at 5/10/15/20 mm/year is descriptive and cannot change the mask.

The notebook restores the exact checksum-verified v11 regenerated checkpoint and re-runs steps 11 and 12 with `--multi_prime` and the unchanged 1.5-radian threshold. **Stop after cell 6 and return to Codex before cell 7.** After that checkpoint, LiCSBAS steps 13 and 14 may run only if the exact-window gates passed. The notebook does not use partial GACOS, manual pair removal, nullification, or relaxed thresholds. Even a pass creates only a shorter-period validation candidate: rates remain relative and require reference sensitivity plus independent date-compatible GNSS or levelling before any deformation, hazard, or risk promotion.


In [ ]:
# 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2 — Install pinned LiCSBAS, licsar_extra, and SNAPHU
import os, sys, subprocess, pathlib

LICSBAS_REPO = pathlib.Path('/content/LiCSBAS')
LICSBAS_COMMIT = 'a145e6c4217b29eb7348ea95630454a099809098'
LICSAR_EXTRA_COMMIT = '557a50811b03a722c7a326c62ea0375949f0b7e4'

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'snaphu', 'gdal-bin', 'libgdal-dev', 'python3-gdal'], check=True)
if not LICSBAS_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/comet-licsar/LiCSBAS.git', str(LICSBAS_REPO)], check=True)
subprocess.run(['git', '-C', str(LICSBAS_REPO), 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', str(LICSBAS_REPO), 'checkout', '--detach', LICSBAS_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(LICSBAS_REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == LICSBAS_COMMIT, (actual_commit, LICSBAS_COMMIT)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'h5py', 'statsmodels', 'astropy', 'cmcrameri', 'bottleneck', 'rasterio', 'requests',
    'beautifulsoup4', 'psutil', 'netCDF4', 'colorcet', 'xarray', 'rioxarray', 'pandas',
    'scipy', 'scikit-learn', 'affine', 'shapely', 'pyproj',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    f'git+https://github.com/comet-licsar/licsar_extra.git@{LICSAR_EXTRA_COMMIT}'
], check=True)
try:
    from osgeo import gdal
except ImportError:
    gdal_version = subprocess.check_output(['gdal-config', '--version'], text=True).strip()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'GDAL=={gdal_version}'], check=True)
    from osgeo import gdal
print('GDAL:', gdal.VersionInfo())
if shutil_path := subprocess.run(['which', 'snaphu'], text=True, stdout=subprocess.PIPE).stdout.strip():
    print('SNAPHU:', shutil_path)
else:
    raise RuntimeError('SNAPHU installation did not produce an executable.')
print('LiCSBAS commit:', actual_commit)
print('licsar_extra commit:', LICSAR_EXTRA_COMMIT)


In [ ]:
# 3 - Frozen configuration and helpers
import datetime, hashlib, json, math, os, re, shutil, subprocess, sys, tarfile, time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path, PurePosixPath

import numpy as np
import rasterio
import requests
from affine import Affine
from pyproj import Transformer
from shapely.geometry import box, shape
from shapely.ops import transform as shapely_transform, unary_union

FRAME = '047D_09652_111009'
TRACK = '47'
START, END = '20200110', '20230611'
DATE_TAG = f'{START}_{END}'
EXPECTED_COMPLETE_PAIRS = 349
EXPECTED_ACQUISITIONS = 76
NLOOK = 5
ROI_BOUNDS = (106.45, -6.54, 107.22, -6.01)
ROI_G_OPTION = f'{ROI_BOUNDS[0]}/{ROI_BOUNDS[2]}/{ROI_BOUNDS[1]}/{ROI_BOUNDS[3]}'
WRAPPED_PRODUCT = 'diff_unfiltered_pha'
UNWRAP_FILTER = 'gold'
UNWRAP_CONSISTENCY_THRESHOLD = 0.3
USE_GACOS = False
LOOP_MODE = 'multi_prime'
LOOP_THRESHOLD_RAD = 1.5

MIN_LCC_IMAGE_FRACTION = 0.90
MIN_LCC_TEMPORAL_SPAN_FRACTION = 0.95
MIN_LCC_EDGE_TO_IMAGE_RATIO = 2.0
MAX_LCC_BRIDGE_INTERFEROGRAMS = 0
MAX_LCC_INTERFEROGRAMS_WITHOUT_TRIANGLE = 0
HISTORICAL_MAX_REJECTION_FRACTION = 0.25

MIN_NORTH_AREA_FRACTION = 0.50
MIN_EACH_NORTH_DISTRICT_AREA_FRACTION = 0.25
MIN_NORTH_LARGEST_PATCH_FRACTION = 0.75
MIN_LOCAL_IMAGE_FRACTION = 0.90
MIN_LOCAL_TEMPORAL_SPAN_FRACTION = 0.95
MIN_LOCAL_EDGE_TO_IMAGE_RATIO = 2.0
MAX_LOCAL_BRIDGES = 0
MAX_LOCAL_EDGES_WITHOUT_TRIANGLE = 0
MIN_NORTH_GRID_COVERAGE_FRACTION = 0.999999
GRID_TOLERANCE_PIXELS = 0.01

BOUNDARY_SHA256 = '7bb4dd0589cec8bd64e8930c94b1835116701b6148f7b7de183e61196593c540'
BOUNDARY_URL = ('https://gis-dpmptsp.jakarta.go.id/arcgis/rest/services/Hosted/'
                'Batas_Administrasi_Kelurahan_DKI_Jakarta/FeatureServer/85/query'
                '?where=1%3D1&outFields=wadmkd%2Cwadmkc%2Cwadmkk&returnGeometry=true&outSR=4326&f=geojson')

WORK = Path('/content/insar_desc47_short_v13')
DRIVE_OUT = Path('/content/drive/MyDrive/InSAR_Risk_Zoning_Project')
ARCHIVE = DRIVE_OUT / 'LiCSBAS_archive_2020_2023'
GEOC_NAME = 'GEOC'
GEOCML_NAME = 'GEOCml5clip'  # Native hard-coded LiCSBAS02to05_unwrap.py output name.
TS_NAME = 'TS_GEOCml5clip_desc47_short_reunw_mp'
RUN_TAG = 'desc47_short_reunw_unfiltered_roi_ml5_mp'
BIN = LICSBAS_REPO / 'bin'
LIB = LICSBAS_REPO / 'LiCSBAS_lib'
CEDA_DATA = f'https://dap.ceda.ac.uk/neodc/comet/data/licsar_products/{TRACK}/{FRAME}'

SOURCE_INVENTORY = ARCHIVE / f'CEDA_{FRAME}_{DATE_TAG}_{WRAPPED_PRODUCT}_inventory.json'
SOURCE_ARCHIVE = ARCHIVE / f'GEOC_{FRAME}_{RUN_TAG}_{DATE_TAG}_source_checkpoint.tar.gz'
SOURCE_ARCHIVE_SHA = ARCHIVE / f'GEOC_{FRAME}_{RUN_TAG}_{DATE_TAG}_source_checkpoint.sha256'
GRID_PREFLIGHT = ARCHIVE / f'GEOC_{FRAME}_{RUN_TAG}_{DATE_TAG}_grid_preflight.json'
GEOCML_ARCHIVE = ARCHIVE / f'{GEOCML_NAME}_{RUN_TAG}_{DATE_TAG}_checkpoint.tar.gz'
GEOCML_ARCHIVE_SHA = ARCHIVE / f'{GEOCML_NAME}_{RUN_TAG}_{DATE_TAG}_checkpoint.sha256'
GEOCML_INVENTORY = ARCHIVE / f'{GEOCML_NAME}_{RUN_TAG}_{DATE_TAG}_inventory.json'
NETWORK_GATE_PATH = DRIVE_OUT / f'jakarta_licsbas_{DATE_TAG}_{RUN_TAG}_network_and_north_gate.json'
DIAGNOSTIC_ARCHIVE = ARCHIVE / f'{TS_NAME}_step12_diagnostics_{DATE_TAG}.tar.gz'
DIAGNOSTIC_ARCHIVE_SHA = ARCHIVE / f'{TS_NAME}_step12_diagnostics_{DATE_TAG}.sha256'
BOUNDARY_LOCAL = WORK / 'dki_kelurahan_official.geojson'
BOUNDARY_CACHE = ARCHIVE / f'dki_kelurahan_official_{BOUNDARY_SHA256[:12]}.geojson'

V11_CHECKPOINT = ARCHIVE / 'GEOCml5clip_desc47_reunw_unfiltered_roi_ml5_mp_20200101_20231231_checkpoint.tar.gz'
V11_CHECKPOINT_SHA_FILE = ARCHIVE / 'GEOCml5clip_desc47_reunw_unfiltered_roi_ml5_mp_20200101_20231231_checkpoint.sha256'
EXPECTED_V11_CHECKPOINT_SHA256 = '0a3d1378234eef757fdb07341abb96dc2934612d4a4124c8282cf40b619565b9'
EXPECTED_V11_CONFIGURATION_FINGERPRINT = '8aa6272c2f629684926140b864361cb15ff3a3c5c988c3bad0cd015b817984f9'
RUN_ROLE = 'preregistered_descending_short_window_inversion_candidate'
POST_INVERSION_GATE_PATH = DRIVE_OUT / f'jakarta_licsbas_{DATE_TAG}_{RUN_TAG}_post_inversion_gate.json'
QC_SUMMARY = DRIVE_OUT / f'jakarta_licsbas_{DATE_TAG}_{RUN_TAG}_qc_summary.json'
MANIFEST_PATH = DRIVE_OUT / f'jakarta_licsbas_{DATE_TAG}_{RUN_TAG}_manifest.json'
RESULTS_ARCHIVE = ARCHIVE / f'{TS_NAME}_results_{DATE_TAG}.tar.gz'
RESULTS_ARCHIVE_SHA = ARCHIVE / f'{TS_NAME}_results_{DATE_TAG}.sha256'

WORK.mkdir(parents=True, exist_ok=True)
ARCHIVE.mkdir(parents=True, exist_ok=True)
run_env = os.environ.copy()
run_env['PYTHONPATH'] = os.pathsep.join([str(LIB), run_env.get('PYTHONPATH', '')]).rstrip(os.pathsep)

def resolve_script(role, preferred, fragments):
    exact = BIN / preferred
    if exact.exists():
        return exact
    candidates = sorted(p for p in BIN.glob('LiCSBAS*.py') if all(x.lower() in p.name.lower() for x in fragments))
    if len(candidates) != 1:
        raise RuntimeError(f'Cannot resolve {role}: {candidates}')
    return candidates[0]

SCRIPTS = {
    'check': resolve_script('check', 'LiCSBAS11_check_unw.py', ('11', 'check_unw')),
    'loop': resolve_script('loop', 'LiCSBAS12_loop_closure.py', ('12', 'loop')),
    'invert': resolve_script('invert', 'LiCSBAS13_sb_inv.py', ('13', 'sb_inv')),
    'vstd': resolve_script('vstd', 'LiCSBAS14_vel_std.py', ('14', 'vel_std')),
    'geotiff': resolve_script('GeoTIFF export', 'LiCSBAS_flt2geotiff.py', ('flt2geotiff',)),
}

def run_script(role, args, required=()):
    cmd = [sys.executable, str(SCRIPTS[role]), *map(str, args)]
    print('RUN:', ' '.join(cmd), flush=True)
    completed = subprocess.run(cmd, cwd=WORK, env=run_env)
    if completed.returncode != 0:
        raise RuntimeError(f'{role} failed with return code {completed.returncode}')
    missing = [str(p) for p in required if not Path(p).exists()]
    if missing:
        raise RuntimeError(f'{role} completed but required output is missing: {missing}')

PAIR_RE = re.compile(r'^\d{8}_\d{8}$')

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def pair_dirs(root):
    root = Path(root)
    return sorted(p for p in root.iterdir() if p.is_dir() and PAIR_RE.fullmatch(p.name)) if root.exists() else []

def geotiff_path(folder, kind):
    matches = sorted(folder.glob(f'*.geo.{kind}.tif'))
    return matches[0] if len(matches) == 1 else None

def source_inventory_on_disk(root):
    pairs = pair_dirs(root)
    missing_cc = [p.name for p in pairs if geotiff_path(p, 'cc') is None]
    missing_phase = [p.name for p in pairs if geotiff_path(p, WRAPPED_PRODUCT) is None]
    return {'pairs': [p.name for p in pairs], 'pair_folders': len(pairs),
            'missing_cc': missing_cc, 'missing_phase': missing_phase}

def regenerated_inventory(root):
    pairs = pair_dirs(root)
    missing = [p.name for p in pairs if not all((p / f'{p.name}.{suffix}').exists() for suffix in ('unw', 'cc', 'conncomp'))]
    return {'pairs': [p.name for p in pairs], 'pair_folders': len(pairs),
            'complete_pairs': len(pairs) - len(missing), 'incomplete_pairs': missing}

def atomic_archive(source_root, items, destination):
    partial = destination.with_suffix(destination.suffix + '.partial')
    if partial.exists():
        partial.unlink()
    subprocess.run(['tar', '-czf', str(partial), '-C', str(source_root), *items], check=True)
    subprocess.run(['tar', '-tzf', str(partial)], check=True, stdout=subprocess.DEVNULL)
    if partial.stat().st_size == 0:
        raise RuntimeError(f'Empty archive: {partial}')
    os.replace(partial, destination)
    return sha256_file(destination)

print(json.dumps({k: str(v) for k, v in SCRIPTS.items()}, indent=2))


In [ ]:
# 4 - Validate the pinned tools and frozen v13 method
for role, script in SCRIPTS.items():
    help_run = subprocess.run([sys.executable, str(script), '-h'], cwd=WORK, env=run_env,
                              text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if help_run.returncode not in (0, 1):
        raise RuntimeError(f'{role} help check failed ({help_run.returncode})')
    print(f'--- {role}: {script.name} ---')
    print('\n'.join(help_run.stdout.splitlines()[:14]))
loop_help = subprocess.run([sys.executable, str(SCRIPTS['loop']), '-h'], cwd=WORK, env=run_env,
                           text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT).stdout
if '--multi_prime' not in loop_help:
    raise RuntimeError('Pinned loop-closure command does not advertise --multi_prime.')
if USE_GACOS:
    raise RuntimeError('Partial GACOS is prohibited in this experiment.')
if (START, END) != ('20200110', '20230611'):
    raise RuntimeError('The preregistered v13 interval changed.')
print('Frozen v13 method verified: exact available descending interval, restored factor-5 checkpoint, 1.5-rad multi-prime loops, then gated inversion.')


## Exact checkpoint restoration

This stage performs only a lightweight live catalogue identity check, restores the approximately 40 MB regenerated v11 checkpoint, verifies its fixed SHA-256 and embedded configuration fingerprint, and acquires the checksum-frozen official DKI boundary. It does not download the 8.45 GB wrapped-phase source archive or regenerate interferograms.


In [ ]:
# 5 - Restore and verify the exact regenerated stack and exact catalogue interval
session = requests.Session()
session.headers.update({'User-Agent': 'Jakarta-InSAR-v13-short-window-preflight'})

if BOUNDARY_CACHE.exists():
    boundary_bytes = BOUNDARY_CACHE.read_bytes()
    if hashlib.sha256(boundary_bytes).hexdigest() != BOUNDARY_SHA256:
        raise RuntimeError('Cached official DKI boundary checksum mismatch.')
else:
    response = session.get(BOUNDARY_URL, timeout=120)
    response.raise_for_status()
    boundary_bytes = response.content
    if hashlib.sha256(boundary_bytes).hexdigest() != BOUNDARY_SHA256:
        raise RuntimeError('Official DKI boundary checksum changed; review it before continuing.')
    partial = BOUNDARY_CACHE.with_suffix(BOUNDARY_CACHE.suffix + '.partial')
    partial.write_bytes(boundary_bytes)
    os.replace(partial, BOUNDARY_CACHE)
BOUNDARY_LOCAL.write_bytes(boundary_bytes)

index_response = session.get(CEDA_DATA + '/', timeout=90)
index_response.raise_for_status()
catalogue_pairs = sorted({
    pair for pair in re.findall(r'(?<!\d)(\d{8}_\d{8})(?!\d)', index_response.text)
    if START <= pair[:8] <= END and START <= pair[-8:] <= END
})
catalogue_images = sorted({date for pair in catalogue_pairs for date in pair.split('_')})
if len(catalogue_pairs) != EXPECTED_COMPLETE_PAIRS or len(catalogue_images) != EXPECTED_ACQUISITIONS:
    raise RuntimeError(f'Exact-window catalogue changed: {len(catalogue_pairs)} pairs/{len(catalogue_images)} dates; expected 349/76.')
if catalogue_images[0] != START or catalogue_images[-1] != END:
    raise RuntimeError(f'Catalogue endpoints changed: {catalogue_images[0]} to {catalogue_images[-1]}.')
expected_pair_set = set(catalogue_pairs)
all_ifgs_expected = catalogue_pairs
all_images_expected = catalogue_images

if not V11_CHECKPOINT.exists():
    raise FileNotFoundError(f'Missing verified regenerated checkpoint in Drive: {V11_CHECKPOINT}')
checkpoint_sha = sha256_file(V11_CHECKPOINT)
if checkpoint_sha != EXPECTED_V11_CHECKPOINT_SHA256:
    raise RuntimeError(f'V11 checkpoint SHA-256 mismatch: {checkpoint_sha}')
if V11_CHECKPOINT_SHA_FILE.exists():
    sidecar_sha = V11_CHECKPOINT_SHA_FILE.read_text(encoding='utf-8').strip().split()[0]
    if sidecar_sha != EXPECTED_V11_CHECKPOINT_SHA256:
        raise RuntimeError(f'V11 checkpoint sidecar disagrees with the preregistered SHA-256: {sidecar_sha}')

geocml_root = WORK / GEOCML_NAME
if geocml_root.exists():
    existing = regenerated_inventory(geocml_root)
    existing_is_complete = (
        existing['pair_folders'] == EXPECTED_COMPLETE_PAIRS
        and existing['complete_pairs'] == EXPECTED_COMPLETE_PAIRS
        and set(existing['pairs']) == expected_pair_set
    )
    if not existing_is_complete:
        if geocml_root.resolve().parent != WORK.resolve():
            raise RuntimeError(f'Refusing to replace an unexpected runtime path: {geocml_root.resolve()}')
        print('Removing an incomplete ephemeral checkpoint extraction before retrying.')
        shutil.rmtree(geocml_root)
if not geocml_root.exists():
    with tarfile.open(V11_CHECKPOINT, 'r:gz') as bundle:
        destination_root = WORK.resolve()
        for member in bundle.getmembers():
            if member.issym() or member.islnk():
                raise RuntimeError(f'Checkpoint contains a prohibited link: {member.name}')
            target = (WORK / member.name).resolve()
            try:
                target.relative_to(destination_root)
            except ValueError as exc:
                raise RuntimeError(f'Checkpoint member escapes the work directory: {member.name}') from exc
        bundle.extractall(WORK, filter='data')

restored = regenerated_inventory(geocml_root)
if (restored['pair_folders'] != EXPECTED_COMPLETE_PAIRS
        or restored['complete_pairs'] != EXPECTED_COMPLETE_PAIRS
        or set(restored['pairs']) != expected_pair_set):
    raise RuntimeError(f'Restored checkpoint does not match the exact 349-pair catalogue: {restored}')
provenance_path = geocml_root / 'reunwrap_provenance.json'
if not provenance_path.exists():
    raise RuntimeError('Restored checkpoint lacks reunwrap_provenance.json.')
reunwrap_provenance = json.loads(provenance_path.read_text(encoding='utf-8'))
if reunwrap_provenance.get('configuration_fingerprint') != EXPECTED_V11_CONFIGURATION_FINGERPRINT:
    raise RuntimeError('Restored checkpoint configuration fingerprint mismatch.')
for key, expected in {'frame': FRAME, 'orbit': 'descending', 'pair_count': 349,
                      'acquisition_count': 76, 'multilook': 5,
                      'wrapped_product': WRAPPED_PRODUCT, 'historical_unwrapped_phase_used': False}.items():
    if reunwrap_provenance.get(key) != expected:
        raise RuntimeError(f'Restored provenance mismatch for {key}: {reunwrap_provenance.get(key)!r} != {expected!r}')
print(json.dumps({'checkpoint_sha256': checkpoint_sha, 'configuration_fingerprint': EXPECTED_V11_CONFIGURATION_FINGERPRINT,
                  'pair_count': len(catalogue_pairs), 'acquisition_count': len(catalogue_images),
                  'first_acquisition': catalogue_images[0], 'last_acquisition': catalogue_images[-1]}, indent=2))


In [ ]:
# 6 - Reproduce steps 11/12 and enforce the exact-window global and North gates
run_script('check', ['-d', GEOCML_NAME, '-t', TS_NAME], required=[WORK / TS_NAME])
run_script('loop', ['-d', GEOCML_NAME, '-t', TS_NAME, '-l', str(LOOP_THRESHOLD_RAD), '--multi_prime'],
           required=[WORK / TS_NAME / 'info' / '12bad_ifg.txt', WORK / TS_NAME / 'results' / 'n_loop_err'])

def data_lines(path):
    return [line.strip() for line in Path(path).read_text(encoding='utf-8', errors='replace').splitlines()
            if line.strip() and not line.lstrip().startswith('#')] if Path(path).exists() else []

def ifg_ids(path):
    result = set()
    for line in data_lines(path):
        match = re.search(r'(?<!\d)(\d{8}_\d{8})(?!\d)', line)
        if not match:
            raise RuntimeError(f'Cannot parse interferogram from {path}: {line}')
        result.add(match.group(1))
    return result

def graph_summary(nodes, edges):
    nodes = sorted(nodes)
    adjacency = {node: set() for node in nodes}
    edge_names = {}
    for pair in edges:
        a, b = pair.split('_')
        adjacency[a].add(b); adjacency[b].add(a)
        edge_names[frozenset((a, b))] = pair
    components, unseen = [], set(nodes)
    while unseen:
        start = min(unseen); stack = [start]; component = set()
        while stack:
            node = stack.pop()
            if node in component:
                continue
            component.add(node); unseen.discard(node); stack.extend(adjacency[node] - component)
        components.append(sorted(component))
    components.sort(key=lambda item: (-len(item), item[0]))
    lcc = components[0] if components else []
    lcc_set = set(lcc)
    lcc_edges = sorted(pair for pair in edges if set(pair.split('_')).issubset(lcc_set))
    disc, low, parent, bridges, clock = {}, {}, {}, [], [0]
    def visit(node):
        disc[node] = low[node] = clock[0]; clock[0] += 1
        for neighbour in sorted(adjacency[node] & lcc_set):
            if neighbour not in disc:
                parent[neighbour] = node; visit(neighbour); low[node] = min(low[node], low[neighbour])
                if low[neighbour] > disc[node]:
                    bridges.append(edge_names[frozenset((node, neighbour))])
            elif parent.get(node) != neighbour:
                low[node] = min(low[node], disc[neighbour])
    if lcc:
        visit(lcc[0])
    without_triangle = []
    for pair in lcc_edges:
        a, b = pair.split('_')
        if not (adjacency[a] & adjacency[b] & lcc_set):
            without_triangle.append(pair)
    return lcc, lcc_edges, sorted(bridges), sorted(without_triangle), components

def all_edge_redundancy(nodes, edges):
    adjacency = {node: set() for node in nodes}
    edge_names = {}
    for pair in edges:
        a, b = pair.split('_')
        adjacency[a].add(b); adjacency[b].add(a)
        edge_names[frozenset((a, b))] = pair
    disc, low, parent, bridges, clock = {}, {}, {}, [], [0]
    def visit(node):
        disc[node] = low[node] = clock[0]; clock[0] += 1
        for neighbour in sorted(adjacency[node]):
            if neighbour not in disc:
                parent[neighbour] = node; visit(neighbour); low[node] = min(low[node], low[neighbour])
                if low[neighbour] > disc[node]:
                    bridges.append(edge_names[frozenset((node, neighbour))])
            elif parent.get(node) != neighbour:
                low[node] = min(low[node], disc[neighbour])
    for node in sorted(nodes):
        if node not in disc:
            visit(node)
    without_triangle = []
    for pair in edges:
        a, b = pair.split('_')
        if not (adjacency[a] & adjacency[b]):
            without_triangle.append(pair)
    return sorted(bridges), sorted(without_triangle)

info_dir = WORK / TS_NAME / 'info'
bad11, bad12 = ifg_ids(info_dir / '11bad_ifg.txt'), ifg_ids(info_dir / '12bad_ifg.txt')
all_ifgs = sorted(p.name for p in pair_dirs(geocml_root))
all_images = sorted({d for pair in all_ifgs for d in pair.split('_')})
if (len(all_ifgs) != EXPECTED_COMPLETE_PAIRS
        or set(all_ifgs) != expected_pair_set
        or len(all_images) != EXPECTED_ACQUISITIONS
        or all_images != catalogue_images):
    raise RuntimeError('Step-12 input identity differs from the frozen 349-pair/76-date catalogue.')
rejected = bad11 | bad12
if rejected - expected_pair_set:
    raise RuntimeError(f'LiCSBAS rejection files contain unknown pairs: {sorted(rejected - expected_pair_set)}')
retained = sorted(set(all_ifgs) - rejected)
lcc, lcc_ifgs, bridges, without_triangle, components = graph_summary(all_images, retained)
requested_days = (datetime.datetime.strptime(END, '%Y%m%d') - datetime.datetime.strptime(START, '%Y%m%d')).days
lcc_days = ((datetime.datetime.strptime(lcc[-1], '%Y%m%d') - datetime.datetime.strptime(lcc[0], '%Y%m%d')).days if len(lcc) > 1 else 0)
global_metrics = {
    'image_fraction': len(lcc) / len(all_images),
    'temporal_span_fraction': lcc_days / requested_days,
    'edge_to_image_ratio': len(lcc_ifgs) / len(lcc),
    'bridges': bridges, 'interferograms_without_triangle': without_triangle,
}
global_failures = []
if global_metrics['image_fraction'] < MIN_LCC_IMAGE_FRACTION: global_failures.append('global image fraction')
if global_metrics['temporal_span_fraction'] < MIN_LCC_TEMPORAL_SPAN_FRACTION: global_failures.append('global temporal span')
if global_metrics['edge_to_image_ratio'] < MIN_LCC_EDGE_TO_IMAGE_RATIO: global_failures.append('global edge/image ratio')
if len(bridges) > MAX_LCC_BRIDGE_INTERFEROGRAMS: global_failures.append('global bridges')
if len(without_triangle) > MAX_LCC_INTERFEROGRAMS_WITHOUT_TRIANGLE: global_failures.append('global triangle coverage')

if not BOUNDARY_LOCAL.exists() or hashlib.sha256(BOUNDARY_LOCAL.read_bytes()).hexdigest() != BOUNDARY_SHA256:
    raise RuntimeError('Checksum-verified boundary preflight is unavailable; rerun the checkpoint-restore cell.')
boundary = json.loads(BOUNDARY_LOCAL.read_bytes())
north_features = [feature for feature in boundary['features'] if 'UTARA' in str(feature['properties'].get('wadmkk', '')).upper()]
district_geometries = defaultdict(list)
for feature in north_features:
    district = str(feature['properties']['wadmkc']).strip()
    district_geometries[district].append(shape(feature['geometry']))
district_geometries = {name: unary_union(parts) for name, parts in district_geometries.items()}
if len(district_geometries) != 6:
    raise RuntimeError(f'Expected six North Jakarta districts, found {sorted(district_geometries)}.')

par = {}
for line in (geocml_root / 'EQA.dem_par').read_text(encoding='utf-8', errors='replace').splitlines():
    fields = line.replace(':', ' ').split()
    if len(fields) >= 2:
        par[fields[0]] = fields[1]
width, length = int(par['width']), int(par['nlines'])
post_lon, post_lat = float(par['post_lon']), float(par['post_lat'])
corner_lon, corner_lat = float(par['corner_lon']), float(par['corner_lat'])
if not all(math.isfinite(value) for value in (post_lon, post_lat, corner_lon, corner_lat)):
    raise RuntimeError('Factor-5 grid contains non-finite georeferencing values.')
if post_lon <= 0 or post_lat >= 0 or width <= 0 or length <= 0:
    raise RuntimeError('Factor-5 grid is not a positive-width north-up longitude/latitude grid.')
grid_transform = Affine(post_lon, 0, corner_lon - post_lon / 2, 0, post_lat, corner_lat - post_lat / 2)
to_utm = Transformer.from_crs(4326, 32748, always_xy=True).transform
district_utm = {name: shapely_transform(to_utm, geom) for name, geom in district_geometries.items()}
grid_left, grid_top = grid_transform * (0, 0)
grid_right, grid_bottom = grid_transform * (width, length)
grid_wgs84 = box(min(grid_left, grid_right), min(grid_bottom, grid_top),
                 max(grid_left, grid_right), max(grid_bottom, grid_top))
grid_utm = shapely_transform(to_utm, grid_wgs84)
district_grid_coverage = {
    name: grid_utm.intersection(geometry).area / geometry.area
    for name, geometry in district_utm.items()
}
grid_coverage_failures = [
    name for name, value in district_grid_coverage.items()
    if value < MIN_NORTH_GRID_COVERAGE_FRACTION
]

cell_district_areas = {}
cell_total_area = {}
for row in range(length):
    for col in range(width):
        left, top = grid_transform * (col, row)
        right, bottom = grid_transform * (col + 1, row + 1)
        cell_utm = shapely_transform(to_utm, box(min(left, right), min(bottom, top), max(left, right), max(bottom, top)))
        areas = {name: cell_utm.intersection(geom).area for name, geom in district_utm.items()}
        areas = {name: area for name, area in areas.items() if area > 0}
        if areas:
            cell_district_areas[(row, col)] = areas
            cell_total_area[(row, col)] = sum(areas.values())
if not cell_total_area:
    raise RuntimeError('North Jakarta has no intersection with the factor-5 grid.')

north_cells = sorted(cell_total_area)
flat_indices = np.array([row * width + col for row, col in north_cells])
valid_edges = np.zeros((len(retained), len(north_cells)), dtype=bool)
for edge_index, pair in enumerate(retained):
    unw = np.fromfile(geocml_root / pair / f'{pair}.unw', dtype='<f4')
    if unw.size != width * length:
        raise RuntimeError(f'Unexpected UNW size for {pair}: {unw.size}.')
    values = unw[flat_indices]
    valid_edges[edge_index] = np.isfinite(values) & (values != 0)
n_loop_err = np.fromfile(WORK / TS_NAME / 'results' / 'n_loop_err', dtype='<f4').reshape(length, width)
n_ifg_noloop_path = WORK / TS_NAME / 'results' / 'n_ifg_noloop'

qualifies = np.zeros((length, width), dtype=bool)
local_summaries = []
for pixel_index, (row, col) in enumerate(north_cells):
    pixel_edges = [pair for pair, valid in zip(retained, valid_edges[:, pixel_index]) if valid]
    local_lcc, local_lcc_edges, local_bridges, local_without_triangle, _ = graph_summary(all_images, pixel_edges)
    all_local_bridges, all_local_without_triangle = all_edge_redundancy(all_images, pixel_edges)
    local_days = ((datetime.datetime.strptime(local_lcc[-1], '%Y%m%d') - datetime.datetime.strptime(local_lcc[0], '%Y%m%d')).days if len(local_lcc) > 1 else 0)
    image_fraction = len(local_lcc) / len(all_images)
    span_fraction = local_days / requested_days
    edge_ratio = len(local_lcc_edges) / len(local_lcc) if local_lcc else 0.0
    passed = (
        image_fraction >= MIN_LOCAL_IMAGE_FRACTION
        and span_fraction >= MIN_LOCAL_TEMPORAL_SPAN_FRACTION
        and edge_ratio >= MIN_LOCAL_EDGE_TO_IMAGE_RATIO
        and len(all_local_bridges) <= MAX_LOCAL_BRIDGES
        and len(all_local_without_triangle) <= MAX_LOCAL_EDGES_WITHOUT_TRIANGLE
        and n_loop_err[row, col] == 0
    )
    qualifies[row, col] = passed
    local_summaries.append({'row': row, 'col': col, 'image_fraction': image_fraction,
                            'temporal_span_fraction': span_fraction, 'edge_to_image_ratio': edge_ratio,
                            'lcc_bridge_count': len(local_bridges),
                            'lcc_edges_without_triangle_count': len(local_without_triangle),
                            'all_finite_retained_bridge_count': len(all_local_bridges),
                            'all_finite_retained_edges_without_triangle_count': len(all_local_without_triangle),
                            'n_loop_err': float(n_loop_err[row, col]), 'qualifies': bool(passed)})

total_area = sum(cell_total_area.values())
passing_area = sum(area for cell, area in cell_total_area.items() if qualifies[cell])
north_fraction = passing_area / total_area
district_total = {name: sum(areas.get(name, 0) for areas in cell_district_areas.values()) for name in district_geometries}
district_passing = {name: sum(areas.get(name, 0) for cell, areas in cell_district_areas.items() if qualifies[cell]) for name in district_geometries}
district_fractions = {name: district_passing[name] / district_total[name] for name in district_geometries}

unseen = {cell for cell in north_cells if qualifies[cell]}
patch_areas = []
while unseen:
    start = min(unseen); stack = [start]; component = set()
    while stack:
        cell = stack.pop()
        if cell in component or cell not in unseen:
            continue
        component.add(cell); unseen.discard(cell)
        row, col = cell
        stack.extend((row + dr, col + dc) for dr in (-1, 0, 1) for dc in (-1, 0, 1)
                     if (dr or dc) and (row + dr, col + dc) in unseen)
    patch_areas.append(sum(cell_total_area[cell] for cell in component))
largest_patch_fraction = max(patch_areas, default=0.0) / passing_area if passing_area else 0.0

north_failures = []
if grid_coverage_failures: north_failures.append('North district grid coverage')
if north_fraction < MIN_NORTH_AREA_FRACTION: north_failures.append('North total area')
if any(value < MIN_EACH_NORTH_DISTRICT_AREA_FRACTION for value in district_fractions.values()): north_failures.append('North district minimum')
if largest_patch_fraction < MIN_NORTH_LARGEST_PATCH_FRACTION: north_failures.append('North connected patch')
sensitivity = []
for total_threshold in (0.40, 0.50, 0.60):
    for district_threshold in (0.15, 0.25, 0.35):
        sensitivity.append({'total_threshold': total_threshold, 'district_threshold': district_threshold,
                            'would_pass': north_fraction >= total_threshold
                            and all(value >= district_threshold for value in district_fractions.values())
                            and largest_patch_fraction >= MIN_NORTH_LARGEST_PATCH_FRACTION})

gate_failures = global_failures + north_failures
network_gate = {
    'gate_basis': 'preregistered_descending_short_window_global_plus_north_local_network',
    'frame': FRAME, 'orbit': 'descending', 'loop_mode': LOOP_MODE, 'loop_threshold_rad': LOOP_THRESHOLD_RAD,
    'total_interferograms': len(all_ifgs), 'total_images': len(all_images),
    'step11_bad_interferograms': len(bad11), 'step12_bad_interferograms': len(bad12),
    'unique_rejected_interferograms': len(rejected),
    'rejection_fraction': len(rejected) / len(all_ifgs),
    'historical_25pct_ceiling_exceeded': len(rejected) / len(all_ifgs) > HISTORICAL_MAX_REJECTION_FRACTION,
    'rejection_fraction_role': 'reported_comparison_not_universal_hard_gate',
    'audit_files': {
        'n_loop_err_present': True,
        'n_ifg_noloop_present': n_ifg_noloop_path.exists(),
        'n_ifg_noloop_role': 'optional LiCSBAS output; custom graph audit independently checks every retained edge for triangle coverage',
    },
    'global_network': {**global_metrics, 'lcc_images': lcc, 'lcc_interferograms': len(lcc_ifgs),
                       'component_count': len(components), 'failures': global_failures},
    'north_jakarta_gate': {
        'denominator': 'projected-area EPSG:32748 intersection with official Jakarta Utara and six wadmkc districts',
        'factor5_grid_bounds_wsen': [grid_left, grid_bottom, grid_right, grid_top],
        'district_grid_coverage_fractions': district_grid_coverage,
        'candidate_cell_count': len(north_cells), 'qualifying_cell_count': int(qualifies.sum()),
        'total_area_m2': total_area, 'qualifying_area_m2': passing_area,
        'qualifying_area_fraction': north_fraction, 'district_area_fractions': district_fractions,
        'largest_8_connected_patch_fraction_of_passing_area': largest_patch_fraction,
        'thresholds': {'min_total_area_fraction': MIN_NORTH_AREA_FRACTION,
                       'min_each_district_area_fraction': MIN_EACH_NORTH_DISTRICT_AREA_FRACTION,
                       'min_largest_patch_fraction': MIN_NORTH_LARGEST_PATCH_FRACTION,
                       'local_image_fraction': MIN_LOCAL_IMAGE_FRACTION,
                       'local_temporal_span_fraction': MIN_LOCAL_TEMPORAL_SPAN_FRACTION,
                       'local_edge_to_image_ratio': MIN_LOCAL_EDGE_TO_IMAGE_RATIO,
                       'max_local_bridges': MAX_LOCAL_BRIDGES,
                       'max_local_edges_without_triangle': MAX_LOCAL_EDGES_WITHOUT_TRIANGLE,
                       'min_north_grid_coverage_fraction': MIN_NORTH_GRID_COVERAGE_FRACTION,
                       'required_n_loop_err': 0},
        'sensitivity_non_decisional': sensitivity, 'failures': north_failures,
    },
    'passed': not gate_failures, 'gate_failures': gate_failures,
    'pass_action': 'Continue to the prospectively gated v13 inversion; outputs remain diagnostic pending post-inversion QC and validation.',
    'promotion_prohibited_without': ['post_inversion_finite_velocity_and_vstd', 'post_inversion_n_gap=0_and_n_loop_err=0',
                                     'unchanged_North_area_district_and_patch_gates', 'uncertainty_sensitivity',
                                     'independent_GNSS_or_levelling_validation'],
}
NETWORK_GATE_PATH.write_text(json.dumps(network_gate, indent=2), encoding='utf-8')
(info_dir / f'{RUN_TAG}_network_and_north_gate.json').write_text(json.dumps(network_gate, indent=2), encoding='utf-8')
(info_dir / f'{RUN_TAG}_north_pixel_ledger.json').write_text(json.dumps(local_summaries, indent=2), encoding='utf-8')
print(json.dumps(network_gate, indent=2))

diagnostic_items = [f'{TS_NAME}/info', f'{TS_NAME}/network', f'{TS_NAME}/12loop',
                    f'{TS_NAME}/results/n_unw', f'{TS_NAME}/results/n_loop_err',
                    f'{GEOCML_NAME}/reunwrap_provenance.json', BOUNDARY_LOCAL.name]
if n_ifg_noloop_path.exists():
    diagnostic_items.append(f'{TS_NAME}/results/n_ifg_noloop')
else:
    print('INFO: optional LiCSBAS results/n_ifg_noloop was not produced; custom all-edge triangle audit is archived instead.')
diagnostic_sha = atomic_archive(WORK, diagnostic_items, DIAGNOSTIC_ARCHIVE)
DIAGNOSTIC_ARCHIVE_SHA.write_text(f'{diagnostic_sha}  {DIAGNOSTIC_ARCHIVE.name}\n', encoding='utf-8')
print(f'VERIFIED diagnostic archive: {DIAGNOSTIC_ARCHIVE} ({DIAGNOSTIC_ARCHIVE.stat().st_size / 1e6:.1f} MB)')
print('SHA-256:', diagnostic_sha)
if gate_failures:
    raise RuntimeError('PREREGISTERED STOP: ' + '; '.join(gate_failures) + '. Diagnostics are archived; do not invert.')
print('EXACT-WINDOW NETWORK + NORTH GATES PASSED. STOP HERE and return to Codex before running cell 7 inversion.')


In [ ]:
# 7 - Run the prospectively gated inversion and compute the strict North candidate
run_script('invert', ['-d', GEOCML_NAME, '-t', TS_NAME, '--mem_size', '4000'],
           required=[WORK / TS_NAME / 'cum.h5', WORK / TS_NAME / 'results' / 'vel',
                     WORK / TS_NAME / 'results' / 'n_gap'])
run_script('vstd', ['-t', TS_NAME], required=[WORK / TS_NAME / 'results' / 'vstd'])

results_dir = WORK / TS_NAME / 'results'
def result_grid(name):
    values = np.fromfile(results_dir / name, dtype='<f4')
    if values.size != width * length:
        raise RuntimeError(f'Unexpected {name} size: {values.size} != {width * length}')
    return values.reshape(length, width)

velocity = result_grid('vel')
velocity_std = result_grid('vstd')
n_gap = result_grid('n_gap')
final_n_loop_err = result_grid('n_loop_err')
finite_results = np.isfinite(velocity) & np.isfinite(velocity_std)
strict_north = qualifies & finite_results & (n_gap == 0) & (final_n_loop_err == 0)

strict_passing_area = sum(area for cell, area in cell_total_area.items() if strict_north[cell])
strict_north_fraction = strict_passing_area / total_area
strict_district_passing = {
    name: sum(areas.get(name, 0) for cell, areas in cell_district_areas.items() if strict_north[cell])
    for name in district_geometries
}
strict_district_fractions = {
    name: strict_district_passing[name] / district_total[name] for name in district_geometries
}
unseen = {cell for cell in north_cells if strict_north[cell]}
strict_patch_areas = []
while unseen:
    start = min(unseen); stack = [start]; component = set()
    while stack:
        cell = stack.pop()
        if cell in component or cell not in unseen:
            continue
        component.add(cell); unseen.discard(cell)
        row, col = cell
        stack.extend((row + dr, col + dc) for dr in (-1, 0, 1) for dc in (-1, 0, 1)
                     if (dr or dc) and (row + dr, col + dc) in unseen)
    strict_patch_areas.append(sum(cell_total_area[cell] for cell in component))
strict_largest_patch_fraction = max(strict_patch_areas, default=0.0) / strict_passing_area if strict_passing_area else 0.0

vstd_sensitivity = []
strict_values = velocity_std[strict_north]
for threshold in (5.0, 10.0, 15.0, 20.0):
    count = int(np.count_nonzero(strict_values <= threshold))
    vstd_sensitivity.append({'threshold_mm_per_year': threshold, 'surviving_pixels': count,
                             'fraction_of_strict_candidate': count / strict_values.size if strict_values.size else 0.0})

post_failures = []
if strict_north_fraction < MIN_NORTH_AREA_FRACTION: post_failures.append('post-inversion North total area')
if any(value < MIN_EACH_NORTH_DISTRICT_AREA_FRACTION for value in strict_district_fractions.values()):
    post_failures.append('post-inversion North district minimum')
if strict_largest_patch_fraction < MIN_NORTH_LARGEST_PATCH_FRACTION:
    post_failures.append('post-inversion North connected patch')
post_inversion_gate = {
    'gate_basis': 'prospective_v13_strict_North_post_inversion_gate',
    'candidate_interval': [START, END], 'frame': FRAME, 'orbit': 'descending',
    'strict_pixel_criteria': ['passes unchanged v11 local network gate', 'finite velocity', 'finite vstd',
                              'n_gap=0', 'n_loop_err=0'],
    'north_candidate_cells': len(north_cells), 'strict_qualifying_cells': int(strict_north.sum()),
    'north_total_area_m2': total_area, 'strict_qualifying_area_m2': strict_passing_area,
    'strict_qualifying_area_fraction': strict_north_fraction,
    'district_area_fractions': strict_district_fractions,
    'largest_8_connected_patch_fraction_of_passing_area': strict_largest_patch_fraction,
    'thresholds': {'min_total_area_fraction': MIN_NORTH_AREA_FRACTION,
                   'min_each_district_area_fraction': MIN_EACH_NORTH_DISTRICT_AREA_FRACTION,
                   'min_largest_patch_fraction': MIN_NORTH_LARGEST_PATCH_FRACTION},
    'vstd_sensitivity_non_decisional': vstd_sensitivity,
    'passed': not post_failures, 'gate_failures': post_failures,
    'scientific_status_if_passed': 'shorter_period_validation_candidate_only',
    'promotion_still_requires': ['reference-area sensitivity', 'independent date-compatible GNSS or levelling validation',
                                  'uncertainty-aware rate interpretation', 'no extrapolation outside the strict mask'],
}
POST_INVERSION_GATE_PATH.write_text(json.dumps(post_inversion_gate, indent=2), encoding='utf-8')
(info_dir / f'{RUN_TAG}_post_inversion_gate.json').write_text(json.dumps(post_inversion_gate, indent=2), encoding='utf-8')
print(json.dumps(post_inversion_gate, indent=2))


In [ ]:
# 8 - Export matched diagnostic layers and the exact North strict candidate mask
layers = ('vel', 'vstd', 'coh_avg', 'n_unw')
exported = {}
for layer in layers:
    source = results_dir / layer
    if not source.exists():
        raise FileNotFoundError(f'Missing LiCSBAS result layer: {source}')
    destination = DRIVE_OUT / f'jakarta_{layer}_{DATE_TAG}_{RUN_TAG}.tif'
    run_script('geotiff', ['-i', source, '-p', geocml_root / 'EQA.dem_par', '-o', destination],
               required=[destination])
    exported[layer] = destination

with rasterio.open(exported['vel']) as template:
    velocity_profile = template.profile.copy()
    exported_velocity = template.read(1)
    if exported_velocity.shape != strict_north.shape:
        raise RuntimeError('Exported velocity grid does not match the strict candidate grid.')
    mask_profile = velocity_profile.copy()
    mask_profile.update(dtype='uint8', count=1, nodata=255, compress='deflate')
    north_domain = np.zeros(strict_north.shape, dtype=bool)
    for cell in north_cells:
        north_domain[cell] = True
    mask_data = np.full(strict_north.shape, 255, dtype='uint8')
    mask_data[north_domain] = 0
    mask_data[strict_north] = 1
    strict_mask_path = DRIVE_OUT / f'jakarta_north_strict_mask_{DATE_TAG}_{RUN_TAG}.tif'
    with rasterio.open(strict_mask_path, 'w', **mask_profile) as destination:
        destination.write(mask_data, 1)
        destination.update_tags(scientific_status='diagnostic_validation_candidate_only',
                                criteria='local network pass; finite vel/vstd; n_gap=0; n_loop_err=0')
    masked_profile = velocity_profile.copy()
    masked_profile.update(dtype='float32', count=1, nodata=np.nan, compress='deflate')
    masked_velocity_path = DRIVE_OUT / f'jakarta_north_velocity_strict_masked_{DATE_TAG}_{RUN_TAG}.tif'
    masked_velocity = np.where(strict_north, exported_velocity, np.nan).astype('float32')
    with rasterio.open(masked_velocity_path, 'w', **masked_profile) as destination:
        destination.write(masked_velocity, 1)
        destination.update_tags(scientific_status='diagnostic_validation_candidate_only', units='mm/year',
                                reference='relative LiCSBAS velocity; not independently validated')
exported['north_strict_mask'] = strict_mask_path
exported['north_velocity_strict_masked'] = masked_velocity_path
print(json.dumps({key: str(value) for key, value in exported.items()}, indent=2))


In [ ]:
# 9 - Validate exports and write complete diagnostic provenance
layer_meta, grid_key = {}, None
for role, path in exported.items():
    with rasterio.open(path) as source:
        data = source.read(1)
        current_grid = (source.width, source.height, str(source.crs), tuple(source.transform))
        if grid_key is None:
            grid_key = current_grid
        elif current_grid != grid_key:
            raise RuntimeError(f'Grid mismatch for {role}: {current_grid} != {grid_key}')
        if role == 'north_strict_mask':
            finite = data != source.nodata
            values = data[finite]
        else:
            finite = np.isfinite(data)
            values = data[finite]
        if role == 'vstd' and (not values.size or np.any(values < 0)):
            raise RuntimeError('vstd is empty or contains negative values.')
        layer_meta[role] = {
            'path': str(path), 'sha256': sha256_file(path), 'bytes': path.stat().st_size,
            'shape': [source.height, source.width], 'crs': str(source.crs),
            'transform': list(source.transform), 'resolution': list(source.res),
            'total_pixels': int(data.size), 'finite_or_domain_pixels': int(finite.sum()),
            'min': float(values.min()) if values.size else None,
            'median': float(np.median(values)) if values.size else None,
            'max': float(values.max()) if values.size else None,
        }

reference_path = info_dir / '12ref.txt'
reference_area = reference_path.read_text(encoding='utf-8', errors='replace').strip() if reference_path.exists() else None
qc = {
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'scientific_status': 'diagnostic_shorter_period_candidate_pending_sensitivity_and_external_validation',
    'run_role': RUN_ROLE, 'frame': FRAME, 'orbit': 'descending', 'start': START, 'end': END,
    'multilook': NLOOK, 'run_tag': RUN_TAG, 'reference_area': reference_area,
    'source_checkpoint': {'path': str(V11_CHECKPOINT), 'sha256': checkpoint_sha,
                          'configuration_fingerprint': EXPECTED_V11_CONFIGURATION_FINGERPRINT},
    'network_gate': network_gate, 'post_inversion_gate': post_inversion_gate,
    'layer_statistics': layer_meta,
    'prohibited_claims': ['validated deformation rate', 'citywide hazard map', 'risk map',
                          'North Jakarta rate without independent validation'],
}
manifest = {
    'created_utc': qc['created_utc'], 'frame': FRAME, 'orbit': 'descending',
    'start': START, 'end': END, 'multilook': NLOOK, 'run_tag': RUN_TAG,
    'run_role': RUN_ROLE, 'licsbas_commit': actual_commit, 'licsar_extra_commit': LICSAR_EXTRA_COMMIT,
    'source_checkpoint': qc['source_checkpoint'],
    'steps': ['restore_verified_v11_regenerated_checkpoint', '11', '12_multi_prime_1.5rad', '13', '14',
              'strict_North_post_inversion_gate'],
    'network_gate': network_gate, 'post_inversion_gate': post_inversion_gate,
    'layers': layer_meta, 'qc_summary': str(QC_SUMMARY),
}
QC_SUMMARY.write_text(json.dumps(qc, indent=2, allow_nan=False), encoding='utf-8')
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, allow_nan=False), encoding='utf-8')
(info_dir / QC_SUMMARY.name).write_text(json.dumps(qc, indent=2, allow_nan=False), encoding='utf-8')
(info_dir / MANIFEST_PATH.name).write_text(json.dumps(manifest, indent=2, allow_nan=False), encoding='utf-8')
print('QC summary:', QC_SUMMARY)
print('Manifest:', MANIFEST_PATH)


In [ ]:
# 10 - Archive all diagnostics, verify the archive, and enforce the final stop/pass decision
archive_items = [f'{TS_NAME}/results', f'{TS_NAME}/info', f'{TS_NAME}/network',
                 f'{GEOCML_NAME}/EQA.dem_par', f'{GEOCML_NAME}/reunwrap_provenance.json',
                 BOUNDARY_LOCAL.name]
results_sha = atomic_archive(WORK, archive_items, RESULTS_ARCHIVE)
RESULTS_ARCHIVE_SHA.write_text(f'{results_sha}  {RESULTS_ARCHIVE.name}\n', encoding='utf-8')
print(f'VERIFIED v13 results archive: {RESULTS_ARCHIVE} ({RESULTS_ARCHIVE.stat().st_size / 1e6:.1f} MB)')
print('SHA-256:', results_sha)
if not post_inversion_gate['passed']:
    raise RuntimeError('PREREGISTERED V13 STOP: ' + '; '.join(post_inversion_gate['gate_failures'])
                       + '. Outputs are archived for audit; do not tune or promote them.')
print('V13 POST-INVERSION GATE PASSED — this is a shorter-period validation candidate only.')
print('Do not promote rates, hazard, or risk until reference sensitivity and independent GNSS/levelling validation are complete.')
